[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/pose_diffusion/station_keeping.ipynb)

# Station Keeping in Gusts

This example computes how far a dynamically positioned vessel wanders from its set point under random wind and wave disturbances. A vessel holds its position and heading with its thrusters while gusts push it off course and proportional control pushes it back toward the set point; because gusts strike the side of the hull harder than the bow, excursions reach much further sideways than forward. Predicting that steady-state spread sets both the clearance radius needed around the vessel and the minimum controller gain that keeps excursions within tolerance.

The vessel's pose error is modelled as a twist bivector from the set point to its pose, paired with antibivector readout lines through `line & error`. Where the term ***duality map*** here means a linear map between a space and its dual readout space, pose spread is a covariance duality map `Twist <- Line` whose variance along a readout line is `line & sigma(line)`. Proportional control together with any steady turning rate defines a linear drift map `Twist.commutator(rate) - gain * Twist` acting on the covariance's twist output and, via `dynamics.adjugate()`, on its line input, so the steady-state balance between gust injection and controller damping reduces to one linear solve batched across controller gains.

In matrix Lie-algebra notation on $\mathfrak{se}(2)$, choosing coordinate order $(v_x, v_y, \omega)$ represents the drift as $F = \operatorname{ad}_\Omega - k I$ with a skew-coupled $3 \times 3$ commutator block $\operatorname{ad}_\Omega$, evolves the symmetric $3 \times 3$ covariance matrix as $\dot P = F P + P F^{\mathsf T} + B B^{\mathsf T}$, and solves the continuous Lyapunov equation $F P + P F^{\mathsf T} + Q = 0$ via the $9 \times 9$ Kronecker product $(I \otimes F + F \otimes I)\,\operatorname{vec} P = -\operatorname{vec} Q$ or Bartels-Stewart reduction.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import numpy as np

from examples.estimation.pose_diffusion import render
from examples.util.animation import save_figure, show_states
from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
# An antibivector line reads a twist through the regressive product.
Line = ga.gatype.antibivector()                     # Antibivector-
# A plane vector reads the displacement of a point through the regressive product.
Plane = ga.gatype.vector()                          # Vector+
Twist = ga.gatype.bivector()                        # Bivector+
Motor = ga.gatype.rotor()                           # Rotor+
Covariance = ga.gatype((Twist, Line))               # Twist+ <- Line-
Dynamics = ga.gatype((Twist, Twist))                # Twist+ <- Twist-
Spread = ga.gatype((Scalar, Plane, Plane))          # Scalar+ <- (Plane+, Plane+)
Gusts = ga.gatype((Twist, Line))                    # Twist+ <- Line-
Quadric = ga.gatype((Plane, Point))                 # Plane+ <- Point-
# Commanded set point at the origin.
origin = mv.w.dual()                                # [] Point-

## Gust covariance and steady-state balance

Gusts act on the hull as white noise mapped from readout lines to body-frame twists through dyads along sideways sway, forward surge, and yaw, each weighted by its disturbance strength. Across an orthonormal frame of readout lines carrying independent unit-variance white noise, the covariance rate injected per second sums the image dyads over that frame. Meanwhile, proportional control at rate `gain` together with any steady turning rate `rate` forms the drift map `Twist.commutator(rate) - gain * Twist`, which acts on the covariance's twist output and through `.adjugate()` on its line input; expanding the unknown covariance on twist-line dyads solves for the steady state where damping balances gust injection.

In matrix Lie-algebra notation, choosing coordinate order $(v_x, v_y, \omega)$ writes the gust map as a $3 \times 3$ matrix $B$ with diffusion rate $Q = B B^{\mathsf T}$ and sets $\dot P = F P + P F^{\mathsf T} + Q = 0$, where $F^{\mathsf T}$ represents the dual pull-back `dynamics.adjugate()`.

In [ ]:
# Body-frame push directions (sideways, forward, and yaw), strengths per square-root second, and controller gain (1/s).
pushes = stack([mv.yw, mv.wx, mv.xy])                                         # [3] Twist+
strengths = np.array([0.5, 0.15, 0.01])
gain = 0.1

In [ ]:
def gust_covariance(gusts: Gusts) -> Covariance:
    """Unit-variance white noise across an orthonormal line frame maps through `gusts` (`[] Twist+ <- Line-`)
    to the covariance rate (`[] Twist+ <- Line-`) added per second."""
    # Orthonormal readout lines carry independent unit-variance white noise, so each adds its gust dyad.
    frame = mv.bivector(np.eye(3)).dual()                                     # [3] Line-
    return (gusts(frame) * (gusts(frame) & Line)).sum(axis=0)                 # [] Twist+ <- Line-


def steady_covariance(dynamics: Dynamics, noise: Covariance) -> Covariance:
    """Balancing drift `dynamics` (`[...] Twist+ <- Twist+`) on both sides against `noise` (`[] Twist+ <- Line-`)
    gives the steady-state covariance (`[...] Twist+ <- Line-`)."""
    # Expanding the covariance on dyads turns its drift rate into a linear map on both twist factors and the line.
    lyapunov = dynamics(Twist) * (Twist & Line) + Twist * (dynamics(Twist) & Line)   # [...] Twist+ <- (Twist+, Twist+, Line-)
    # Matching the line input against -noise solves for both twist factors; Line.dual() feeds a line in as a twist.
    return lyapunov.lstsq(-noise)(Line.dual())                                # [...] Twist+ <- Line-


gusts = (pushes * strengths * (pushes & Line)).sum(axis=-1)                   # [] Twist+ <- Line-
still = mv.xy * 0.0                                                           # [] Twist+
controller = Twist.commutator(still) - gain * Twist                           # [] Twist+ <- Twist+
noise = gust_covariance(gusts)                                                # [] Twist+ <- Line-
steady = steady_covariance(controller, noise)                                 # [] Twist+ <- Line-
print("gust covariance rate per second, a duality map from lines to twists:\n", noise.coefficient_table())

## Fleet diffusion while holding heading and turning

Simulating ensembles of vessels from the set point tests the Lyapunov prediction directly in time. At each step every vessel's twist error advances under the controller and receives an independent white-noise gust scaled by the square root of `dt`, while the predicted covariance advances by `dynamics(predicted) + predicted(dynamics.adjugate()) + noise`. Comparing a fleet holding a fixed heading against an identical fleet turning at one revolution per minute shows how steady rotation mixes sideways gusts into forward excursions, rounding the position confidence quadric.

In [ ]:
# Fleet size, run duration and timestep in seconds, frame stride, random seeds, and plot half-width in metres.
n_vessels = 2000
seconds, dt = 60.0, 0.2
n_steps = int(seconds / dt)
n_stride = 3
n_frames = n_steps // n_stride
seed, turning_seed = 0, 1
turning_rate = mv.xy * (2 * np.pi / 60)                                       # [] Twist+
extent, sigmas = 5.0, 2.0
frame_duration_ms = 40

In [ ]:
def position_ellipsoid(sigma: Covariance, sigmas: float) -> Quadric:
    """Pushing `sigma` (`[...] Twist+ <- Line-`) to the set point's second moment gives the implicit confidence
    quadrics (`[...] Plane+ <- Point-`) at `sigmas` standard deviations."""
    shift = Twist.commutator(origin)                                          # [] Point- <- Twist+
    readout = shift.adjugate()                                                # [] Line- <- Plane+
    moment = origin * (Plane & origin) + shift(sigma(readout))                # [...] Point- <- Plane+
    return moment.solve(Point) - (1 + sigmas**2) * mv.w * (mv.w & Point)      # [...] Plane+ <- Point-


def fleet_diffusion(
    dynamics: Dynamics,
    gusts: Gusts,
    noise: Covariance,
    limit: Covariance,
    dt: float,
    n_steps: int,
    n_stride: int,
    n_vessels: int,
    seed: int,
) -> Iterator[tuple[Motor, Covariance, Covariance]]:
    """Yields `(poses, predicted, limit)` (`[n_vessels] Motor+`, `[] Twist+ <- Line-`, `[] Twist+ <- Line-`) every
    `n_stride` steps across `n_steps` Euler-Maruyama steps of a fleet starting at the set point."""
    rng = np.random.default_rng(seed)
    errors = mv.bivector(np.zeros((n_vessels, 3)))                            # [n_vessels] Twist+
    predicted = noise * 0.0                                                   # [] Twist+ <- Line-
    for step_index in range(n_steps):
        # Integrated white noise scales as sqrt(dt), while drift acts on both twist output and line input.
        white = mv.bivector(rng.normal(size=(n_vessels, 3))).dual() * np.sqrt(dt)   # [n_vessels] Line-
        errors = errors + dynamics(errors) * dt + gusts(white)                # [n_vessels] Twist+
        growth = dynamics(predicted) + predicted(dynamics.adjugate()) + noise   # [] Twist+ <- Line-
        predicted = predicted + growth * dt                                   # [] Twist+ <- Line-
        if (step_index + 1) % n_stride == 0:
            yield (errors * 0.5).exp(), predicted, limit


turning_controller = Twist.commutator(turning_rate) - gain * Twist            # [] Twist+ <- Twist+
turning_steady = steady_covariance(turning_controller, noise)                 # [] Twist+ <- Line-
ellipsoid_of = partial(position_ellipsoid, sigmas=sigmas)
streams = (
    fleet_diffusion(controller, gusts, noise, steady, dt, n_steps, n_stride, n_vessels, seed),
    fleet_diffusion(turning_controller, gusts, noise, turning_steady, dt, n_steps, n_stride, n_vessels, turning_seed),
)
frames = show_states(
    zip(*streams),
    render.clouds_animation,
    extent,
    ellipsoid_of,
    name="station_keeping",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

In the left panel the fleet holds a fixed heading and spreads into a tall sideways ellipse; in the right panel turning at one revolution per minute rotates sideways excursions into the quieter forward direction so the cloud rounds out and shrinks along its major axis. Blue arrows mark a sample of vessels along their headings, the shaded pale-red ellipse shows the solved steady-state two-standard-deviation region, and the solid red curve shows the time-integrated covariance growing from zero onto that steady boundary.

In [ ]:
holding_final, turning_final = frames[-1]
save_figure(render.settled_figure([holding_final, turning_final], extent, ellipsoid_of), "station_keeping_turning");

## Choosing the controller gain

Raising the controller gain tightens station keeping at the cost of stronger thruster action. Because the Lyapunov balance is a linear solve, evaluating a batch of gains solves every steady-state covariance at once. Pulling displacement-measuring planes back through the set-point motion `Twist.commutator(origin)` yields the symmetric position variance form on planes, whose principal half-widths identify the smallest gain keeping the two-standard-deviation excursion within a 2 m clearance.

In matrix notation, projecting the $3 \times 3$ pose covariance to position forms $\Sigma = J P J^{\mathsf T}$ with the $2 \times 3$ displacement Jacobian $J$ at the reference point.

In [ ]:
# Controller gains (1/s) swept in one batch and maximum two-standard-deviation excursion in metres.
n_gains = 60
gains = np.linspace(0.02, 0.5, n_gains)
tolerance = 2.0

In [ ]:
def position_spread(sigma: Covariance) -> Spread:
    """Pulling displacement-measuring planes back through the set-point motion maps `sigma`
    (`[...] Twist+ <- Line-`) to the symmetric position variance form (`[...] Scalar+ <- (Plane+, Plane+)`)."""
    # A twist displaces the set point by their commutator; its adjugate pulls displacement planes back to twist lines.
    shift = Twist.commutator(origin)                                          # [] Point- <- Twist+
    readout = shift.adjugate()                                                # [] Line- <- Plane+
    return readout & sigma(readout)                                           # [...] Scalar+ <- (Plane+, Plane+)


gain_controllers = Twist.commutator(still) - mv.scalar(gains[:, None]) * Twist   # [n_gains] Twist+ <- Twist+
spreads = position_spread(steady_covariance(gain_controllers, noise))         # [n_gains] Scalar+ <- (Plane+, Plane+)
save_figure(render.envelope_figure(gains, spreads, tolerance), "station_keeping_gain");

The red curve traces the major-axis two-standard-deviation half-width, the blue curve traces the minor-axis half-width, the horizontal dashed line marks the 2 m clearance, and the vertical dotted line marks the minimum gain meeting it. Along the hull's side the two-standard-deviation excursion equals the sideways gust strength divided by the square root of twice the gain, requiring a gain of 0.125 per second (an 8 s time constant), while leaving a wide margin forward.

In [ ]:
# checks
# Steady-state covariances zero the Lyapunov drift rate, time integration converges onto them,
# empirical fleet twists match the steady covariance, and turning shrinks the major-axis spread.
frame = mv.bivector(np.eye(3)).dual()                                           # [3] Line-
poses, predicted, _ = holding_final
steady_growth = controller(steady) + steady(controller.adjugate()) + noise      # [] Twist+ <- Line-
assert (frame[:, None] & steady_growth(frame[None, :])).abs().max() < 1e-10
steady_scale = (frame[:, None] & steady(frame[None, :])).abs().max()            # [] Scalar+
assert (frame[:, None] & (predicted - steady)(frame[None, :])).abs().max() < 5e-3 * steady_scale
errors = poses.log() * 2                                                        # [n_vessels] Twist+
empirical = (errors * (errors & Line)).mean(axis=0)                             # [] Twist+ <- Line-
assert (frame[:, None] & (empirical - steady)(frame[None, :])).abs().max() < 0.12 * steady_scale
# Sideways variance at fixed heading equals `strengths[0]**2 / (2 * gain)`, and turning reduces the peak variance.
holding_variances = position_spread(steady).eigvalsh()                          # [2] Scalar+
turning_variances = position_spread(turning_steady).eigvalsh()                  # [2] Scalar+
assert (holding_variances[-1] - strengths[0]**2 / (2 * gain)).abs() < 1e-8
assert turning_variances[-1] < holding_variances[-1]